# Quotes and Trades Contracts and Monitoring

Notebook refactorizado.

Ya no actua como lanzador de agentes.

Su funcion ahora es:
- ver el contrato de descarga,
- documentar comandos y rutas de prueba,
- dejar templates de produccion,
- y visualizar artefactos de certificacion para quotes y referencias relacionadas.

## Estado Actual

Este notebook sustituye al flujo antiguo que mezclaba:
- Agent01,
- Agent02 laxo `031`,
- Agent02 estricto `032`,
- y rutas antiguas `C:\TSIS_Data\data\quotes_p95`.

Operacion actual recomendada:
- lanzar Agent01/02/03 desde terminal,
- usar notebooks solo para consulta, contrato y visualizacion.

In [ ]:
from pathlib import Path
import json
import pandas as pd

CONTRACT_FILE = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\contracts_polygon_download.yaml")
CONTRACT_VIEWER = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\030_show_contract_polygon_download.py")

TEST_RUN_ID = "20260312_quotes_lifecycle3_test"
TEST_RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit") / TEST_RUN_ID
TEST_QUOTES_ROOT = Path(r"D:\quotes\__pruebas__\lifecycle_prod")
TEST_TASKS_CSV = TEST_RUN_DIR / "inputs" / "tasks_lifecycle3.csv"
TEST_TASKS_META = TEST_RUN_DIR / "inputs" / "tasks_lifecycle3_meta.json"


PROD_RUN_ID = "20260313_quotes_prod_full_12133_clean"
PROD_RUN_DIR = PROD_RUN_DIR_TEMPLATE / PROD_RUN_ID
print("PROD_RUN_DIR:", PROD_RUN_DIR, "exists", PROD_RUN_DIR.exists())
print("download_events_current exists:", (PROD_RUN_DIR / "download_events_current.csv").exists())
print("download_live_status exists:", (PROD_RUN_DIR / "download_live_status.json").exists())

PROD_QUOTES_ROOT = Path(r"D:\quotes")
PROD_RUN_DIR_TEMPLATE = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit")

print("CONTRACT_FILE:", CONTRACT_FILE, "exists", CONTRACT_FILE.exists())
print("CONTRACT_VIEWER:", CONTRACT_VIEWER, "exists", CONTRACT_VIEWER.exists())
print("TEST_RUN_DIR:", TEST_RUN_DIR)
print("TEST_QUOTES_ROOT:", TEST_QUOTES_ROOT)
print("PROD_QUOTES_ROOT:", PROD_QUOTES_ROOT)

CONTRACT_FILE: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\contracts_polygon_download.yaml exists True
CONTRACT_VIEWER: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\030_show_contract_polygon_download.py exists True
TEST_RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test
TEST_QUOTES_ROOT: D:\quotes\__pruebas__\lifecycle_prod
PROD_QUOTES_ROOT: D:\quotes


In [2]:
if CONTRACT_VIEWER.exists() and CONTRACT_FILE.exists():
    DATASET_TO_VIEW = "quotes"
    exec(CONTRACT_VIEWER.read_text(encoding="utf-8"), globals())
else:
    print("Contrato o viewer no disponibles")

Contract file: C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\contracts_polygon_download.yaml
version: 1 | profile: lax_realtime_coverage


,dataset,enabled,priority,target_path,expected_file_name,partition_pattern,min_cols,min_rows_per_file,coverage_unit,success_criteria
0,quotes,True,1,quotes,quotes.parquet,{ticker}/year={year}/month={month}/day={day}/q...,"timestamp, bid_price, ask_price",1,day,file_exists_and_rows_gt_0
1,trades_ticks_2005_2025,True,2,trades_ticks_2005_2025,market.parquet,{ticker}/year={year}/month={month}/day={day}/m...,"timestamp, price, size",1,day,file_exists_and_rows_gt_0



Detalle dataset: quotes
{
  "enabled": true,
  "priority": 1,
  "target_path": "quotes",
  "expected_file_name": "quotes.parquet",
  "partition_pattern": "{ticker}/year={year}/month={month}/day={day}/quotes.parquet",
  "partition_keys": [
    "ticker",
    "year",
    "month",
    "day"
  ],
  "minimum_required_columns": [
    "timestamp",
    "bid_price",
    "ask_price"
  ],
  "non_empty_required": true,
  "min_rows_per_file": 1,
  "soft_quality_rules": {
    "bid_ask_sanity": {
      "enabled": true,
      "expression": "bid_price <= ask_price",
      "allow_violations": true
    },
    "non_negative_prices": {
      "enabled": true,
      "columns": [
        "bid_price",
        "ask_price"
      ],
      "allow_violations": true
    }
  },
  "coverage": {
    "coverage_unit": "day",
    "expected_files_per_day": 1,
    "success_criteria": "file_exists_and_rows_gt_0",
    "completeness_metric": "present_days / expected_days_official_window"
  },
  "retry": {
    "enabled": true,


## Comandos de Prueba que se Han Usado

Corrida de prueba actual:
- `RUN_ID = 20260312_quotes_lifecycle3_test`
- `QuotesRoot = D:\quotes\__pruebas__\lifecycle_prod`
- `TasksCsv = C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3.csv`

Artefactos principales:
- `C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test`
- `quotes_agent_strict_events_current.csv`
- `retry_queue_quotes_strict_current.csv`
- `live_status_quotes_strict.json`
- `agent03_outputs\run_summary.json`

In [3]:
from pathlib import Path

DOWNLOADER_SCRIPT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py")
AGENT_RUNNERS_DIR = Path(globals().get("AGENT_RUNNERS_DIR", r"C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents"))
AGENT02_PS1 = Path(globals().get("AGENT02_PS1", AGENT_RUNNERS_DIR / "run_agent02_quotes_strict_loop.ps1"))
AGENT03_LIVE_PS1 = Path(globals().get("AGENT03_LIVE_PS1", AGENT_RUNNERS_DIR / "run_agent03_live_fast.ps1"))
AGENT03_COMPACT_PS1 = Path(globals().get("AGENT03_COMPACT_PS1", AGENT_RUNNERS_DIR / "run_agent03_monitor_compact.ps1"))

print(rf'''python {DOWNLOADER_SCRIPT} ^
  --csv "{TEST_TASKS_CSV}" ^
  --output "{TEST_QUOTES_ROOT}" ^
  --concurrent 24 ^
  --run-id "{TEST_RUN_ID}" ^
  --run-dir "{TEST_RUN_DIR}" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -QuotesRoot "{TEST_QUOTES_ROOT}" ^
  -MaxFiles 50000 ^
  -SleepSec 15 ^
  -ResetState''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_LIVE_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -IntervalSec 10''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_COMPACT_PS1}" ^
  -RunId "{TEST_RUN_ID}" ^
  -SleepSec 30''')

python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py ^
  --csv "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\inputs\tasks_lifecycle3.csv" ^
  --output "D:\quotes\__pruebas__\lifecycle_prod" ^
  --concurrent 24 ^
  --run-id "20260312_quotes_lifecycle3_test" ^
  --run-dir "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "20260312_quotes_lifecycle3_test" ^
  -QuotesRoot "D:\quotes\__pruebas__\lifecycle_prod" ^
  -MaxFiles 50000 ^
  -SleepSec 15 ^
  -ResetState

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent03_live_fast.ps1" ^
  -RunId "20260312_quotes_lifecycle3_test" ^
  -IntervalSec 10

powershell -Exec

## Comandos Preparados de Produccion

Quotes:
- raiz objetivo: `D:\quotes`
- mismo esquema de `RUN_ID` y `RUN_DIR`
- misma sesion extendida `04:00:00 -> 20:00:00 ET`

Trades:
- este notebook no deja un launcher de produccion de trades porque el flujo operativo actual no esta estandarizado aqui.
- se mantiene como referencia de contrato y monitoreo, no como orchestrator de `quotes + trades`.

In [4]:
PROD_RUN_ID = "YYYYMMDD_quotes_session_prod_01"
PROD_RUN_DIR = PROD_RUN_DIR_TEMPLATE / PROD_RUN_ID
PROD_TASKS_CSV = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\quotes_tasks_prod.csv")

print(rf'''python {DOWNLOADER_SCRIPT} ^
  --csv "{PROD_TASKS_CSV}" ^
  --output "{PROD_QUOTES_ROOT}" ^
  --concurrent 24 ^
  --run-id "{PROD_RUN_ID}" ^
  --run-dir "{PROD_RUN_DIR}" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT02_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -QuotesRoot "{PROD_QUOTES_ROOT}" ^
  -MaxFiles 50000 ^
  -SleepSec 15''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_LIVE_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -IntervalSec 10''')

print()
print(rf'''powershell -ExecutionPolicy Bypass -File "{AGENT03_COMPACT_PS1}" ^
  -RunId "{PROD_RUN_ID}" ^
  -SleepSec 30''')

python C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py ^
  --csv "C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\quotes_tasks_prod.csv" ^
  --output "D:\quotes" ^
  --concurrent 24 ^
  --run-id "YYYYMMDD_quotes_session_prod_01" ^
  --run-dir "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\YYYYMMDD_quotes_session_prod_01" ^
  --task-batch-size 500 ^
  --session-start 04:00:00 ^
  --session-end 20:00:00

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent02_quotes_strict_loop.ps1" ^
  -RunId "YYYYMMDD_quotes_session_prod_01" ^
  -QuotesRoot "D:\quotes" ^
  -MaxFiles 50000 ^
  -SleepSec 15

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent03_live_fast.ps1" ^
  -RunId "YYYYMMDD_quotes_session_prod_01" ^
  -IntervalSec 10

powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\run_agent03_monitor_compact.ps1" ^
  -RunId "

## Visualizacion de Resultados y Artefactos

Esta parte reemplaza a los launchers viejos de `031` y `032` dentro del notebook.

In [5]:
events_current = TEST_RUN_DIR / "quotes_agent_strict_events_current.csv"
retry_current = TEST_RUN_DIR / "retry_queue_quotes_strict_current.csv"
live_status = TEST_RUN_DIR / "live_status_quotes_strict.json"
run_summary = TEST_RUN_DIR / "agent03_outputs" / "run_summary.json"
causes_by_ticker = TEST_RUN_DIR / "agent03_outputs" / "causes_by_ticker.csv"

for p in [events_current, retry_current, live_status, run_summary, causes_by_ticker]:
    print(p, "exists", p.exists())

if live_status.exists():
    print("\nLIVE STATUS")
    print(json.dumps(json.loads(live_status.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

if events_current.exists():
    ev = pd.read_csv(events_current)
    print("\nEVENTS CURRENT rows:", len(ev))
    if "severity" in ev.columns:
        print(ev["severity"].value_counts(dropna=False))
    display(ev.head(20))

if retry_current.exists():
    rq = pd.read_csv(retry_current)
    print("\nRETRY CURRENT rows:", len(rq))
    display(rq.head(20))

if run_summary.exists():
    print("\nRUN SUMMARY")
    print(json.dumps(json.loads(run_summary.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))

if causes_by_ticker.exists():
    cbt = pd.read_csv(causes_by_ticker)
    print("\nCAUSES BY TICKER rows:", len(cbt))
    display(cbt.head(20))

C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\quotes_agent_strict_events_current.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\retry_queue_quotes_strict_current.csv exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\live_status_quotes_strict.json exists True
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\agent03_outputs\run_summary.json exists False
C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_lifecycle3_test\agent03_outputs\causes_by_ticker.csv exists False

LIVE STATUS
{
  "run_id": "20260312_quotes_lifecycle3_test",
  "updated_utc": "2026-03-12T14:02:00.388761+00:00",
  "probe_root": "D:\\quotes\\__pruebas__\\lifecycle_prod",
  "max_files": 50000,
  "files_discovered_total": 1695,
  "files_pending": 401,
  "files_processed_total_state"

,file,rows,severity,issues,warns,action,crossed_rows,crossed_ratio_pct,negative_price_rows,missing_required_cols,dtype_mismatches,ask_integer_pct,bid_integer_pct,ask_eq_round_bid_pct,processed_at_utc,run_id
0,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1493,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,7,0.468855,0,[],[],6.831882,4.621567,6.630944,2026-03-12 14:01:58.112045+00:00,20260312_quotes_lifecycle3_test
1,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,225,PASS,[],[],count_coverage,0,0.000000,0,[],[],2.666667,0.444444,2.666667,2026-03-12 14:01:58.113872+00:00,20260312_quotes_lifecycle3_test
2,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1259,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,2,0.158856,0,[],[],5.003971,1.906275,5.003971,2026-03-12 14:01:58.115725+00:00,20260312_quotes_lifecycle3_test
3,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,9700,SOFT_FAIL,[],['crossed_rows_present_but_under_threshold'],warn_and_retry_later,2,0.020619,0,[],[],1.536082,5.886598,1.536082,2026-03-12 14:01:58.118402+00:00,20260312_quotes_lifecycle3_test
4,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,260,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.384615,0.769231,0.384615,2026-03-12 14:01:58.120017+00:00,20260312_quotes_lifecycle3_test
5,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,1601,PASS,[],[],count_coverage,0,0.000000,0,[],[],1.374141,2.186134,1.311680,2026-03-12 14:01:58.121839+00:00,20260312_quotes_lifecycle3_test
6,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,659,PASS,[],[],count_coverage,0,0.000000,0,[],[],7.435508,9.711684,7.283763,2026-03-12 14:01:58.123427+00:00,20260312_quotes_lifecycle3_test
7,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,399,PASS,[],[],count_coverage,0,0.000000,0,[],[],7.017544,2.756892,7.017544,2026-03-12 14:01:58.124988+00:00,20260312_quotes_lifecycle3_test
8,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,442,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.452489,0.452489,0.226244,2026-03-12 14:01:58.126618+00:00,20260312_quotes_lifecycle3_test
9,D:\quotes\__pruebas__\lifecycle_prod\INLF\year...,273,PASS,[],[],count_coverage,0,0.000000,0,[],[],0.732601,1.098901,0.366300,2026-03-12 14:01:58.128085+00:00,20260312_quotes_lifecycle3_test



RETRY CURRENT rows: 0


,file,severity,issues,warns,action,processed_at_utc


**PROBLEMA**  en cada batch rehacía en memoria el  download_events_current.csv completo con concat + sort_values + drop_duplicates sobre más de 2.1 millones de  filas, y lo cambié por un merge incremental por task_key que solo deduplica el batch nuevo y sobrescribe las claves afectadas.

```
processed=2101000/3066263
batch_done=2101000/3066263
Traceback (most recent call last):
  File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py", line 718, in main
    return asyncio.run(run(cfg))
           ~~~~~~~~~~~^^^^^^^^^^
  File "C:\Users\AlexJ\AppData\Local\Programs\Python\Python313\Lib\asyncio\runners.py", line 195, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "C:\Users\AlexJ\AppData\Local\Programs\Python\Python313\Lib\asyncio\runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "C:\Users\AlexJ\AppData\Local\Programs\Python\Python313\Lib\asyncio\base_events.py", line 725, in run_until_complete
    return future.result()
           ~~~~~~~~~~~~~^^
  File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py", line 622, in run
    flush_batch()
    ~~~~~~~~~~~^^
  File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py", line 585, in flush_batch
    merged_current = save_current_merge(events_current_csv, merged_current, new_events)
  File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\download_quotes.py", line 168, in save_current_merge
    cur = cur.sort_values("processed_at_utc").drop_duplicates(subset=["task_key"], keep="last")
          ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AlexJ\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\frame.py", line 7207, in sort_values
    indexer = nargsort(
        k, kind=kind, ascending=ascending, na_position=na_position, key=key
    )
  File "C:\Users\AlexJ\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\sorting.py", line 445, in nargsort
    indexer = np.concatenate([indexer, nan_idx])
numpy._core._exceptions._ArrayMemoryError: Unable to allocate 16.0 MiB for an array with shape (2101100,) and data type int64
```


In [1]:
from pathlib import Path
import pandas as pd

run_dir = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean")
cur = pd.read_csv(run_dir / "download_events_current.csv")

print(cur["status"].value_counts(dropna=False))
print("rows_current:", len(cur))
print("tasks_terminal_ok:", int(cur["status"].isin(["DOWNLOADED_OK", "EMPTY_CONFIRMED"]).sum()))

status
DOWNLOADED_OK       1627256
DOWNLOADED_EMPTY     473741
DOWNLOAD_FAIL             3
Name: count, dtype: int64
rows_current: 2101000
tasks_terminal_ok: 1627256


**Eso confirma dos cosas:**

1. sí está reanudando bastante

- rows_current = 2101000
- DOWNLOADED_OK = 1627256
- DOWNLOAD_FAIL = 3

2. pero hay una inconsistencia importante con los vacíos

- en current tienes:
    - DOWNLOADED_EMPTY = 473741
- pero el código de quotes que vimos usa como terminal/resume:
    - DOWNLOADED_OK
    - EMPTY_CONFIRMED

O sea:

- DOWNLOADED_EMPTY no coincide con EMPTY_CONFIRMED
- si eso sigue así en el script en ejecución, esos vacíos no se estarían saltando en resume como deberían

Tu cuenta lo delata:

- tasks_terminal_ok = 1627256
- solo cuenta DOWNLOADED_OK
- no está contando los 473741 vacíos como terminales

Conclusión:

- reanuda bien los DOWNLOADED_OK
- probablemente no está tratando DOWNLOADED_EMPTY como terminal de resume
- eso es un bug de semántica de estado, no de los parquets en D:\quotes

La cuenta buena, si los vacíos fueran terminales, sería:

- 1627256 + 473741 = 2100997
- y quedarían solo 3 fallos fuera

Si quieres, el siguiente paso correcto es que te parchee download_quotes.py para que DOWNLOADED_EMPTY entre
también en:

- TERMINAL_OK
- RESUME_SKIP_STATUSES

Así el resume quedaría coherente con lo que ya hay en download_events_current.csv.


---

In [8]:
from pathlib import Path
import pandas as pd

run_dir = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean")
cur = pd.read_csv(run_dir / "download_events_current.csv")

cur["status"] = cur["status"].astype(str)

mask = cur["status"].isin(["EMPTY_CONFIRMED", "DOWNLOADED_EMPTY"])
empty_df = cur.loc[mask].copy()

print("status counts:")
print(empty_df["status"].value_counts(dropna=False))

print("\nrows_total_empty_like:", len(empty_df))

display(
    empty_df[["task_key", "ticker", "date", "status", "rows", "pages", "empty_rechecks", "processed_at_utc"]]
    .sort_values(["status", "processed_at_utc"], ascending=[True, False])
    .head(5)
)


status counts:
status
DOWNLOADED_EMPTY    473741
Name: count, dtype: int64

rows_total_empty_like: 473741


,task_key,ticker,date,status,rows,pages,empty_rechecks,processed_at_utc
2100969,PLRZ|2024-11-28,PLRZ,2024-11-28,DOWNLOADED_EMPTY,0,1,1,2026-03-18T03:35:32.081903+00:00
2100914,PLRZ|2025-04-18,PLRZ,2025-04-18,DOWNLOADED_EMPTY,0,1,1,2026-03-18T03:35:06.660652+00:00
2100910,PLSE|2016-04-18,PLSE,2016-04-18,DOWNLOADED_EMPTY,0,1,1,2026-03-18T03:35:06.531555+00:00
2100902,PLSE|2016-05-06,PLSE,2016-05-06,DOWNLOADED_EMPTY,0,1,1,2026-03-18T03:35:05.358083+00:00
2100893,PLSE|2016-04-19,PLSE,2016-04-19,DOWNLOADED_EMPTY,0,1,1,2026-03-18T03:35:04.059425+00:00


In [7]:
# Y si quieres solo el conteo exacto:

from pathlib import Path
import pandas as pd

run_dir = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean")
cur = pd.read_csv(run_dir / "download_events_current.csv")

print(
    cur["status"]
    .astype(str)
    .value_counts(dropna=False)
    .reindex(["EMPTY_CONFIRMED", "DOWNLOADED_EMPTY"], fill_value=0))

status
EMPTY_CONFIRMED          0
DOWNLOADED_EMPTY    473741
Name: count, dtype: int64


**DOWNLOADED_EMPTY significa esto:**

- la tarea ticker,date sí se ejecutó
- la llamada a Polygon respondió sin resultados útiles para esa ventana/sesión
- el run la registró como vacía
- y ahora, con el parche, se considera terminal para resume



Lo importante es separar dos cosas:

1. DOWNLOADED_EMPTY plausible

- de verdad no hay quotes útiles ese día

2. DOWNLOADED_EMPTY sospechoso

- sí parece haber mercado y aun así el feed quedó vacío

La forma correcta de investigarlo es muestrear casos y contrastarlos con contexto local:

- ohlcv_1m
- ohlcv_daily


Ya está corregido en download_quotes.py. He corregido un bug de reanudación en Agent01 de quotes: el run guardaba las tareas vacías como DOWNLOADED_EMPTY,
  pero el resume solo saltaba EMPTY_CONFIRMED, y lo arreglé haciendo que DOWNLOADED_EMPTY también se considere
  estado terminal y de skip.

Ahora quotes trata como terminales de resume:

- DOWNLOADED_OK
- DOWNLOADED_EMPTY
- EMPTY_CONFIRMED

Eso arregla exactamente la inconsistencia que encontraste en download_events_current.csv.

Impacto práctico:

- al relanzar Agent01 de quotes con el mismo RUN_ID
- ya no debería reintentar esas 473741 tareas vacías
- se saltarán igual que los DOWNLOADED_OK

No cambia los parquets en D:\quotes.
Cambia solo la semántica de reanudación del run.

Si el proceso que relanzaste sigue corriendo con el código viejo:

- hay que reiniciarlo para que coja este parche

Después, la comprobación buena es:

- ver que tasks_already_ok sube incluyendo también los DOWNLOADED_EMPTY
- no solo los DOWNLOADED_OK

**explorador ticker/date con contexto ohlcv_1m/daily, igual que hicimos en trades_ticks**

In [10]:
from pathlib import Path

RUN_ID = '20260313_quotes_prod_full_12133_clean'
RUN_DIR = Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit') / RUN_ID
QUOTES_ROOT = Path(r'D:\quotes')

print("RUN_DIR:", RUN_DIR)
print("QUOTES_ROOT:", QUOTES_ROOT)
print("download_events_current exists:", (RUN_DIR / "download_events_current.csv").exists())

RUN_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean
QUOTES_ROOT: D:\quotes
download_events_current exists: True


In [14]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean")
CURRENT_CSV = RUN_DIR / "download_events_current.csv"
DAILY_ROOT = Path(r"D:\ohlcv_daily")

cur = pd.read_csv(CURRENT_CSV)
cur["ticker"] = cur["ticker"].astype(str).str.upper().str.strip()
cur["date"] = pd.to_datetime(cur["date"], errors="coerce")
cur = cur[cur["status"] == "DOWNLOADED_EMPTY"].copy()
cur = cur.dropna(subset=["ticker", "date"]).reset_index(drop=True)

def load_daily_years(ticker: str, center_date: pd.Timestamp) -> pd.DataFrame:
    years = sorted({center_date.year - 1, center_date.year, center_date.year + 1})
    parts = []
    for year in years:
        fp = DAILY_ROOT / f"ticker={ticker}" / f"year={year:04d}" / f"day_aggs_{ticker}_{year:04d}.parquet"
        if not fp.exists():
            continue
        try:
            pf = pq.ParquetFile(fp)
            cols = [c for c in ["date", "o", "h", "l", "c", "v"] if c in pf.schema.names]
            df = pf.read(columns=cols).to_pandas()
            parts.append(df)
        except Exception:
            continue
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    return out.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

flags = []
for row in cur[["ticker", "date"]].itertuples(index=False):
    daily = load_daily_years(row.ticker, row.date)
    has_daily_exact = False
    if not daily.empty:
        has_daily_exact = bool((daily["date"] == row.date).any())
    flags.append(has_daily_exact)

out = cur.copy()
out["has_daily_exact"] = flags
out = out[out["has_daily_exact"]].copy()

print("DOWNLOADED_EMPTY with exact daily bar:", len(out))
print("tickers:", out["ticker"].nunique())

display(
    out[["ticker", "date", "status", "rows", "pages", "empty_rechecks", "processed_at_utc", "has_daily_exact"]]
    .sort_values(["ticker", "date"])
    .head(100)
)

DOWNLOADED_EMPTY with exact daily bar: 3614
tickers: 55


,ticker,date,status,rows,pages,empty_rechecks,processed_at_utc,has_daily_exact
16687,AIEV,2025-04-22,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:01:16.482560+00:00,True
16950,AIEV,2025-04-23,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:01:25.873163+00:00,True
16917,AIEV,2025-04-24,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:01:25.481366+00:00,True
16711,AIEV,2025-04-25,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:01:16.952713+00:00,True
16688,AIEV,2025-04-28,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:01:16.497253+00:00,True
...,...,...,...,...,...,...,...,...
24740,AIMD,2022-02-18,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:06:36.321370+00:00,True
24753,AIMD,2022-02-22,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:06:36.408177+00:00,True
24993,AIMD,2022-02-23,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:06:38.775974+00:00,True
24785,AIMD,2022-02-24,DOWNLOADED_EMPTY,0,1,1,2026-03-13T02:06:36.572538+00:00,True


In [15]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

RUN_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean")
CURRENT_CSV = RUN_DIR / "download_events_current.csv"
DAILY_ROOT = Path(r"D:\ohlcv_daily")

cur = pd.read_csv(CURRENT_CSV)
cur["ticker"] = cur["ticker"].astype(str).str.upper().str.strip()
cur["date"] = pd.to_datetime(cur["date"], errors="coerce")
cur = cur[cur["status"] == "DOWNLOADED_EMPTY"].copy()
cur = cur.dropna(subset=["ticker", "date"]).reset_index(drop=True)

def load_daily_years(ticker: str, center_date: pd.Timestamp) -> pd.DataFrame:
    years = sorted({center_date.year - 1, center_date.year, center_date.year + 1})
    parts = []
    for year in years:
        fp = DAILY_ROOT / f"ticker={ticker}" / f"year={year:04d}" / f"day_aggs_{ticker}_{year:04d}.parquet"
        if not fp.exists():
            continue
        try:
            pf = pq.ParquetFile(fp)
            cols = [c for c in ["date", "o", "h", "l", "c", "v"] if c in pf.schema.names]
            df = pf.read(columns=cols).to_pandas()
            parts.append(df)
        except Exception:
            continue
    if not parts:
        return pd.DataFrame()
    out = pd.concat(parts, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    return out.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

flags = []
for row in cur[["ticker", "date"]].itertuples(index=False):
    daily = load_daily_years(row.ticker, row.date)
    has_daily_exact = False
    if not daily.empty:
        has_daily_exact = bool((daily["date"] == row.date).any())
    flags.append(has_daily_exact)

out = cur.copy()
out["has_daily_exact"] = flags
out = out[out["has_daily_exact"]].copy()

summary = (
    out.groupby("ticker", dropna=False)
        .agg(
            empty_with_daily_exact=("ticker", "size"),
            date_min=("date", "min"),
            date_max=("date", "max"),
        )
        .reset_index()
        .sort_values(["empty_with_daily_exact", "ticker"], ascending=[False, True])
)

print("tickers with DOWNLOADED_EMPTY and exact daily bar:", len(summary))
display(summary.head(100))

tickers with DOWNLOADED_EMPTY and exact daily bar: 55


,ticker,empty_with_daily_exact,date_min,date_max
25,CURR,391,2021-12-31,2023-08-08
49,NESR,353,2023-04-28,2024-10-21
53,OPGN,213,2024-08-20,2025-07-17
20,COEP,205,2021-12-31,2022-10-28
26,DBGI,166,2024-12-18,2025-08-19
41,LTCH,155,2023-08-10,2024-03-21
1,AIMD,131,2021-12-31,2022-08-08
13,BSFC,123,2024-12-20,2025-06-20
50,NMHI,110,2025-01-15,2025-06-24
22,CRKN,105,2025-03-05,2025-08-04


Sí, esta salida es valiosa y sí señala casos sospechosos.

  Qué demuestra:

  - tienes 55 tickers con DOWNLOADED_EMPTY
  - pero además con barra daily exacta en esa misma fecha

  Eso significa:

  - sí hubo al menos actividad agregada ese día
  - pero el downloader de quotes recibió vacío para esas tareas

  Eso no prueba automáticamente bug, pero sí eleva la sospecha bastante.

  La interpretación correcta es:

  1. No todos los DOWNLOADED_EMPTY son iguales

  - muchos vacíos pueden ser plausibles
  - estos 55 tickers ya no son “vacío obvio”, porque daily confirma mercado ese día

  2. Los más sospechosos son los que acumulan muchos casos

  - CURR: 391
  - NESR: 353
  - OPGN: 213
  - COEP: 205
  - DBGI: 166
  - etc.

  Esos merecen auditoría primero.

  3. Qué puede significar técnicamente

  - hubo barra daily pero no quotes en el feed consultado
  - o el filtro horario/sesión dejó fuera los quotes
  - o hay una desalineación entre dataset de quotes y dataset daily
  - o Polygon no tiene cobertura histórica de quotes tan completa como la de aggs para esos ticker/date

  4. No todos los casos pesan igual

  - un ticker con 1-5 casos puede ser ruido
  - un ticker con 100+ casos ya parece patrón estructural

  Mi lectura científica:

  - esto no demuestra corrupción del run
  - pero sí demuestra que hay un subconjunto de DOWNLOADED_EMPTY que no deberíamos aceptar sin revisión
  - esos 55 tickers son una buena primera lista roja

  El siguiente paso útil sería una tabla más fina con:

  - ticker
  - empty_with_daily_exact
  - % empty_with_daily_exact / total_empty
  - y una muestra de fechas por ticker

  Y luego, para los top 10:

  - abrir explorador ticker/date
  - mirar si también hay ohlcv_1m exacto ese día

In [16]:
from pathlib import Path

RUN_ID = '20260313_quotes_prod_full_12133_clean'
RUN_DIR = Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit') / RUN_ID
CURRENT_CSV = RUN_DIR / 'download_events_current.csv'
DAILY_ROOT = Path(r'D:\ohlcv_daily')
TOP_N = 100
TOP_CASES = 200

SCRIPT_208 = Path(r'C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\00_data_certification\208_quotes_empty_suspicious_review.py')

exec(SCRIPT_208.read_text(encoding='utf-8'), globals())

=== QUOTES EMPTY SUSPICIOUS REVIEW ===
{'tickers_suspicious': 55, 'rows_empty_total': 474032, 'rows_empty_with_daily_exact': 3614}


,ticker,empty_with_daily_exact,date_min,date_max,empty_total,pct_empty_with_daily_exact,review_score
0,NESR,353,2023-04-28,2024-10-21,468,75.427350,266.258547
1,CURR,391,2021-12-31,2023-08-08,728,53.708791,210.001374
2,OPGN,213,2024-08-20,2025-07-17,343,62.099125,132.271137
3,DBGI,166,2024-12-18,2025-08-19,212,78.301887,129.981132
4,NMHI,110,2025-01-15,2025-06-24,125,88.000000,96.800000
5,BSFC,123,2024-12-20,2025-06-20,163,75.460123,92.815951
6,COEP,205,2021-12-31,2022-10-28,524,39.122137,80.200382
7,CRKN,105,2025-03-05,2025-08-04,151,69.536424,73.013245
8,LTCH,155,2023-08-10,2024-03-21,331,46.827795,72.583082
9,ATXI,81,2025-03-19,2025-07-18,171,47.368421,38.368421



=== SAMPLE CASES TOP SUSPICIOUS ===


,ticker,date,status,rows,pages,empty_rechecks,processed_at_utc
0,ATXI,2025-03-19,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T10:24:26.676033+00:00
1,ATXI,2025-03-20,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T10:24:22.637422+00:00
2,ATXI,2025-03-21,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T10:25:06.431077+00:00
3,ATXI,2025-03-24,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T10:24:23.571121+00:00
4,ATXI,2025-03-25,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T10:24:24.263197+00:00
...,...,...,...,...,...,...,...
195,BSFC,2025-06-09,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T16:59:21.655546+00:00
196,BSFC,2025-06-10,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T16:59:22.060417+00:00
197,BSFC,2025-06-11,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T16:59:18.279776+00:00
198,BSFC,2025-06-12,DOWNLOADED_EMPTY,0,1.0,1,2026-03-13T16:59:43.994174+00:00


**Relanzi agente 01**

```bash
PS C:\Users\AlexJ> powershell -ExecutionPolicy Bypass -File "C:\TSIS_Data\v1\backtest_SmallCaps\scripts\agents\launch_prod_agent01_quotes.ps1" -RunId "20260313_quotes_prod_full_12133_clean" -CsvPath "C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_prod_full_12133\inputs\tasks_quotes_prod.csv"
run_id=20260313_quotes_prod_full_12133_clean
run_dir=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260313_quotes_prod_full_12133_clean
output_root=D:\quotes
csv=C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygon_realtime_audit\20260312_quotes_prod_full_12133\inputs\tasks_quotes_prod.csv
tasks_total=3066263
tasks_already_ok=2100997
tasks_to_process=965266
processed=100/965266
processed=200/965266
processed=300/965266
processed=400/965266
processed=500/965266
batch_done=497/965266
processed=500/965266
processed=600/965266
processed=700/965266
processed=800/965266
processed=900/965266
batch_done=997/965266
processed=1000/965266
processed=1100/965266
processed=1200/965266
processed=1300/965266
processed=1400/965266
```